# MyoLab-AI — GRABMyo ETL bằng remote streaming
## Primary-4 gesture × Forearm-16 channels × Feature Set 14

Notebook này là **standalone** và được thiết kế để không tải ZIP GRABMyo 9+ GB về máy hoặc Google Drive.

## Chiến lược dữ liệu

```text
PhysioNet WFDB record
        │
        ├── đọc trực tiếp từng record qua mạng
        ├── chỉ lấy 16 kênh forearm cần dùng
        ├── giữ tín hiệu trong RAM cho đúng 1 record
        ├── window + feature extraction
        ├── checkpoint feature theo session × subject
        └── giải phóng tín hiệu; không lưu .dat/.hea
```

Notebook chỉ tải các metadata nhỏ như `RECORDS`, `SHA256SUMS.txt`, `MotionSequence.txt`, `subject-info.csv` để kiểm kê và provenance. Tín hiệu được đọc bằng `wfdb.rdrecord(..., pn_dir=...)` trực tiếp từ PhysioNet.

## Frozen modeling view

- PhysioNet dataset version: `GRABMyo 1.1.0`.
- Subjects: `43`.
- Sessions/days: `3`.
- Trials per gesture per session: `7`.
- Four classes comparable với Mendeley primary view:
  - `gesture17 → rest`
  - `gesture16 → hand_close`
  - `gesture12 → wrist_flexion`
  - `gesture11 → wrist_extension`
- Primary channels: `F1..F16`, tức 16 forearm channels, indices `0..15` trong WFDB record.
- Sampling rate: `2048 Hz`.
- Record length: `5 s = 10240 samples`.
- Window: `200 ms → 410 samples` bằng round-half-up.
- Hop: `100 ms → 205 samples`.
- Expected windows per record: `48`.
- Expected selected records: `43 × 3 × 4 × 7 = 3612`.
- Expected windows before QC: `3612 × 48 = 173376`.
- Split frozen theo subject: `34 train / 9 validation`; mọi session của cùng subject luôn ở cùng partition.
- Không training, không scaler fitting, không test set, không impute `NaN/Inf`.

## Output chính

```text
/content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/
├── manifests/
│   ├── grabmyo-primary4-record-catalog.csv
│   ├── grabmyo-subject-split-manifest.csv
│   ├── grabmyo-record-coverage.csv
│   ├── grabmyo-feature-exclusion-report.csv
│   ├── grabmyo-etl-evidence.json
│   └── grabmyo-etl-final-gate.json
└── outputs/
    ├── grabmyo-primary4-forearm16-fall14.npz
    └── grabmyo-feature-column-manifest.json
```

> Đây là engineering/research artifact, không phải clinical model và không được dùng để đưa ra quyết định điều trị.


## 1 — Environment setup

**Input:** một Colab/runtime có Internet.  
**Output:** các package cần thiết, đặc biệt là `wfdb`.


In [1]:
# === CELL 1: Environment setup ===
from __future__ import annotations

import importlib.util
import subprocess
import sys

_REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "requests": "requests",
    "wfdb": "wfdb",
}

_missing = [
    pip_name
    for import_name, pip_name in _REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if _missing:
    print("[INFO] Installing missing packages:", _missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *_missing]
    )

print("[PASS] Required Python packages are available.")


[INFO] Installing missing packages: ['wfdb']
[PASS] Required Python packages are available.


## 2 — Imports, Drive mount, paths, governance và frozen contracts

**Input:** Google Drive access.  
**Output:** toàn bộ contract của ETL được khóa trước khi đọc tín hiệu.


In [2]:
# === CELL 2: Imports, Drive mount, paths, governance and frozen contracts ===
from __future__ import annotations

import csv
import gzip
import hashlib
import json
import math
import os
import platform
import shutil
import time
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import requests
import scipy
import scipy.stats
import wfdb
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, files  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

# -----------------------------------------------------------------------------
# 1. Governance
# -----------------------------------------------------------------------------
TRAINING_ALLOWED = False
MODEL_FITTING_ALLOWED = False
SCALER_FITTING_ALLOWED = False
TEST_SET_OPENED = False
POOLED_TRAINING_ALLOWED = False
CLINICAL_USE_ALLOWED = False

assert TRAINING_ALLOWED is False
assert MODEL_FITTING_ALLOWED is False
assert SCALER_FITTING_ALLOWED is False
assert TEST_SET_OPENED is False
assert POOLED_TRAINING_ALLOWED is False
assert CLINICAL_USE_ALLOWED is False

# -----------------------------------------------------------------------------
# 2. Dataset identity and remote source
# -----------------------------------------------------------------------------
DATASET_ID = "grabmyo-physionet-v1.1.0"
DATASET_SOURCE = "PhysioNet"
DATASET_VERSION = "1.1.0"
DATASET_DOI = "10.13026/89dm-f662"
DATASET_VIEW_ID = "grabmyo-primary4-forearm16-fall14-v1"
PHYSIONET_DATABASE_DIR = f"grabmyo/{DATASET_VERSION}"
PHYSIONET_FILE_BASE_URL = (
    f"https://physionet.org/files/grabmyo/{DATASET_VERSION}"
)

# Full GRABMyo contract
EXPECTED_SUBJECT_IDS = tuple(range(1, 44))
EXPECTED_SESSION_IDS = (1, 2, 3)
EXPECTED_TRIAL_IDS = tuple(range(1, 8))
EXPECTED_FS_HZ = 2048
EXPECTED_RECORD_SECONDS = 5
EXPECTED_SIGNAL_SAMPLES = EXPECTED_FS_HZ * EXPECTED_RECORD_SECONDS
EXPECTED_HEADER_CHANNEL_COUNT = 32

# Comparable primary-4 view. Correct order follows MotionSequence.txt.
GESTURE_ID_TO_CANONICAL = {
    17: "rest",
    16: "hand_close",
    12: "wrist_flexion",
    11: "wrist_extension",
}
PRIMARY_GESTURE_IDS = (17, 16, 12, 11)
PRIMARY_CLASS_ORDER = (
    "rest",
    "hand_close",
    "wrist_flexion",
    "wrist_extension",
)
assert tuple(GESTURE_ID_TO_CANONICAL[g] for g in PRIMARY_GESTURE_IDS) == PRIMARY_CLASS_ORDER

# Official channel layout: forearm channels are WFDB indices 0..15.
PRIMARY_CHANNEL_INDICES = tuple(range(16))
PRIMARY_CHANNEL_IDS = tuple(f"F{i}" for i in range(1, 17))
assert len(PRIMARY_CHANNEL_INDICES) == len(PRIMARY_CHANNEL_IDS) == 16

# Frozen subject split. Validation list is explicit to avoid version-dependent drift.
VALIDATION_SUBJECT_IDS = (12, 13, 25, 26, 29, 33, 34, 35, 41)
TRAIN_SUBJECT_IDS = tuple(
    subject_id
    for subject_id in EXPECTED_SUBJECT_IDS
    if subject_id not in set(VALIDATION_SUBJECT_IDS)
)
assert len(TRAIN_SUBJECT_IDS) == 34
assert len(VALIDATION_SUBJECT_IDS) == 9
assert set(TRAIN_SUBJECT_IDS).isdisjoint(VALIDATION_SUBJECT_IDS)

# -----------------------------------------------------------------------------
# 3. Window and feature contracts
# -----------------------------------------------------------------------------
def round_half_up(value: float) -> int:
    return int(
        Decimal(str(value)).quantize(
            Decimal("1"), rounding=ROUND_HALF_UP
        )
    )

WINDOW_MS = 200
HOP_MS = 100
WINDOW_SAMPLES = round_half_up(EXPECTED_FS_HZ * WINDOW_MS / 1000.0)
HOP_SAMPLES = round_half_up(EXPECTED_FS_HZ * HOP_MS / 1000.0)
EXPECTED_WINDOWS_PER_RECORD = (
    (EXPECTED_SIGNAL_SAMPLES - WINDOW_SAMPLES) // HOP_SAMPLES + 1
)

assert WINDOW_SAMPLES == 410
assert HOP_SAMPLES == 205
assert EXPECTED_WINDOWS_PER_RECORD == 48

PREPROCESSING_POLICY_ID = "per-record-channel-mean-v1"
WINDOWING_VERSION = "grabmyo-windowing-200ms-100ms-round-half-up-v1"
LABEL_MAPPING_VERSION = "grabmyo-primary4-label-map-v1"
CHANNEL_POLICY_ID = "grabmyo-forearm16-primary-v1"
SPLIT_VERSION = "grabmyo-subject-holdout-34-9-v1"
FEATURE_SET_VERSION = "feature-set-14.v1.0.0"

FEATURE_ORDER = (
    "rms",
    "mav",
    "skewness_unbiased",
    "kurtosis_fisher_unbiased",
    "max_signed",
    "min_signed",
    "std_sample_ddof1",
    "mean",
    "spectral_min_power",
    "spectral_max_power",
    "spectral_std_power_ddof1",
    "mdf_hz",
    "mnf_hz",
    "spectral_entropy_bits",
)
TD8_FEATURES = FEATURE_ORDER[:8]
SP6_FEATURES = FEATURE_ORDER[8:]
NO_MOMENTS12_FEATURES = tuple(
    name for name in FEATURE_ORDER
    if name not in {"skewness_unbiased", "kurtosis_fisher_unbiased"}
)

EXPECTED_RECORD_COUNT = (
    len(EXPECTED_SUBJECT_IDS)
    * len(EXPECTED_SESSION_IDS)
    * len(PRIMARY_GESTURE_IDS)
    * len(EXPECTED_TRIAL_IDS)
)
EXPECTED_WINDOW_COUNT_BEFORE_QC = (
    EXPECTED_RECORD_COUNT * EXPECTED_WINDOWS_PER_RECORD
)
EXPECTED_FEATURE_DIM = len(PRIMARY_CHANNEL_IDS) * len(FEATURE_ORDER)

assert EXPECTED_RECORD_COUNT == 3612
assert EXPECTED_WINDOW_COUNT_BEFORE_QC == 173376
assert EXPECTED_FEATURE_DIM == 224

# -----------------------------------------------------------------------------
# 4. Execution mode
# -----------------------------------------------------------------------------
# FULL streams 3612 selected records (~2.3 GB raw transfer, no raw persistence).
# PILOT streams a tiny deterministic subset for code verification only.
RUN_MODE = os.environ.get("GRABMYO_ETL_RUN_MODE", "FULL").strip().upper()
if RUN_MODE not in {"FULL", "PILOT"}:
    raise ValueError("GRABMYO_ETL_RUN_MODE must be FULL or PILOT")

FORCE_RECOMPUTE_SHARDS = False
DOWNLOAD_FINAL_NPZ_TO_BROWSER = False
MAX_REMOTE_ATTEMPTS = 4
REMOTE_BACKOFF_SECONDS = (2, 5, 15, 30)

if RUN_MODE == "PILOT":
    ACTIVE_SUBJECT_IDS = (1, 12)
    ACTIVE_SESSION_IDS = (1,)
    ACTIVE_GESTURE_IDS = PRIMARY_GESTURE_IDS
    ACTIVE_TRIAL_IDS = (1, 2)
else:
    ACTIVE_SUBJECT_IDS = EXPECTED_SUBJECT_IDS
    ACTIVE_SESSION_IDS = EXPECTED_SESSION_IDS
    ACTIVE_GESTURE_IDS = PRIMARY_GESTURE_IDS
    ACTIVE_TRIAL_IDS = EXPECTED_TRIAL_IDS

# -----------------------------------------------------------------------------
# 5. Storage: only derived artifacts persist
# -----------------------------------------------------------------------------
if IN_COLAB:
    RUNTIME_ROOT = Path("/content/data") / DATASET_ID
else:
    RUNTIME_ROOT = Path.cwd() / ".grabmyo-etl-runtime" / DATASET_ID

DRIVE_DATASET_ROOT = (
    Path("/content/drive/MyDrive/MyoLab-AI-data") / DATASET_ID
    if Path("/content/drive/MyDrive").exists()
    else Path.cwd() / "grabmyo-etl-output" / DATASET_ID
)

METADATA_DIR = RUNTIME_ROOT / "metadata"
CATALOG_DIR = RUNTIME_ROOT / "catalog"
SHARD_DIR = RUNTIME_ROOT / "shards"
EXCLUSION_SHARD_DIR = RUNTIME_ROOT / "exclusions-by-shard"
AUDIT_SHARD_DIR = RUNTIME_ROOT / "record-audit-by-shard"
MANIFEST_DIR = RUNTIME_ROOT / "manifests"
OUTPUT_DIR = RUNTIME_ROOT / "outputs"
LOG_DIR = RUNTIME_ROOT / "logs"

DRIVE_MANIFEST_DIR = DRIVE_DATASET_ROOT / "manifests"
DRIVE_OUTPUT_DIR = DRIVE_DATASET_ROOT / "outputs"
# Derived checkpoints only; no raw .dat/.hea files are stored here.
DRIVE_SHARD_DIR = DRIVE_DATASET_ROOT / "cache" / "etl-shards-v1"
DRIVE_EXCLUSION_SHARD_DIR = DRIVE_DATASET_ROOT / "cache" / "etl-exclusions-v1"
DRIVE_AUDIT_SHARD_DIR = DRIVE_DATASET_ROOT / "cache" / "etl-record-audit-v1"

for directory in [
    METADATA_DIR, CATALOG_DIR, SHARD_DIR, EXCLUSION_SHARD_DIR,
    AUDIT_SHARD_DIR, MANIFEST_DIR, OUTPUT_DIR, LOG_DIR,
    DRIVE_MANIFEST_DIR, DRIVE_OUTPUT_DIR, DRIVE_SHARD_DIR,
    DRIVE_EXCLUSION_SHARD_DIR, DRIVE_AUDIT_SHARD_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RECORDS_PATH = METADATA_DIR / "RECORDS"
SHA256SUMS_PATH = METADATA_DIR / "SHA256SUMS.txt"
MOTION_SEQUENCE_PATH = METADATA_DIR / "MotionSequence.txt"
SUBJECT_INFO_PATH = METADATA_DIR / "subject-info.csv"
LICENSE_PATH = METADATA_DIR / "LICENSE.txt"

RECORD_CATALOG_CSV = MANIFEST_DIR / "grabmyo-primary4-record-catalog.csv"
SUBJECT_SPLIT_CSV = MANIFEST_DIR / "grabmyo-subject-split-manifest.csv"
SUBJECT_SPLIT_JSON = MANIFEST_DIR / "grabmyo-subject-split-manifest.json"
RECORD_COVERAGE_CSV = MANIFEST_DIR / "grabmyo-record-coverage.csv"
EXCLUSION_REPORT_CSV = MANIFEST_DIR / "grabmyo-feature-exclusion-report.csv"
ETL_EVIDENCE_JSON = MANIFEST_DIR / "grabmyo-etl-evidence.json"
FINAL_GATE_JSON = MANIFEST_DIR / "grabmyo-etl-final-gate.json"
FINAL_NPZ = OUTPUT_DIR / "grabmyo-primary4-forearm16-fall14.npz"
COLUMN_MANIFEST_JSON = OUTPUT_DIR / "grabmyo-feature-column-manifest.json"

np.random.seed(4301)

print("=" * 92)
print("[CELL 2] GRABMyo REMOTE-STREAM ETL CONFIGURATION")
print("=" * 92)
print(f"Run mode                      : {RUN_MODE}")
print(f"Dataset                       : {DATASET_ID}")
print(f"PhysioNet directory           : {PHYSIONET_DATABASE_DIR}")
print(f"Active subjects               : {len(ACTIVE_SUBJECT_IDS)}")
print(f"Active sessions               : {ACTIVE_SESSION_IDS}")
print(f"Active gesture IDs            : {ACTIVE_GESTURE_IDS}")
print(f"Active trials                 : {ACTIVE_TRIAL_IDS}")
print(f"Full selected records         : {EXPECTED_RECORD_COUNT}")
print(f"Full windows before QC        : {EXPECTED_WINDOW_COUNT_BEFORE_QC}")
print(f"Window / hop samples          : {WINDOW_SAMPLES} / {HOP_SAMPLES}")
print(f"Windows per record            : {EXPECTED_WINDOWS_PER_RECORD}")
print(f"Primary feature dimension     : {EXPECTED_FEATURE_DIM}")
print(f"Raw ZIP persistence           : False")
print(f"Raw WFDB persistence          : False")
print(f"Runtime root                  : {RUNTIME_ROOT}")
print(f"Persistent root               : {DRIVE_DATASET_ROOT}")
print(f"Training allowed              : {TRAINING_ALLOWED}")
print("[PASS] Frozen contracts initialized.")


Mounted at /content/drive
[CELL 2] GRABMyo REMOTE-STREAM ETL CONFIGURATION
Run mode                      : FULL
Dataset                       : grabmyo-physionet-v1.1.0
PhysioNet directory           : grabmyo/1.1.0
Active subjects               : 43
Active sessions               : (1, 2, 3)
Active gesture IDs            : (17, 16, 12, 11)
Active trials                 : (1, 2, 3, 4, 5, 6, 7)
Full selected records         : 3612
Full windows before QC        : 173376
Window / hop samples          : 410 / 205
Windows per record            : 48
Primary feature dimension     : 224
Raw ZIP persistence           : False
Raw WFDB persistence          : False
Runtime root                  : /content/data/grabmyo-physionet-v1.1.0
Persistent root               : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0
Training allowed              : False
[PASS] Frozen contracts initialized.


## 3 — Common helpers

**Input:** contracts từ Cell 2.  
**Output:** atomic I/O, HTTP metadata retrieval, hashing và JSON safety.


In [3]:
# === CELL 3: Common helpers ===
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_json_atomic(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def copy_atomic(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    shutil.copy2(source, temporary)
    if temporary.stat().st_size != source.stat().st_size:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Copy size mismatch: {source} -> {destination}")
    temporary.replace(destination)


def write_dataframe_atomic(dataframe: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    compression = "gzip" if path.name.endswith(".csv.gz") else None
    dataframe.to_csv(temporary, index=False, compression=compression)
    temporary.replace(path)


def build_http_session() -> requests.Session:
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        raise_on_status=False,
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"User-Agent": "MyoLab-AI-GRABMyo-ETL/1.0"})
    return session


HTTP_SESSION = build_http_session()


def fetch_small_file(remote_name: str, destination: Path) -> Path:
    url = f"{PHYSIONET_FILE_BASE_URL}/{remote_name}"
    response = HTTP_SESSION.get(url, timeout=(20, 120))
    response.raise_for_status()
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.write_bytes(response.content)
    temporary.replace(destination)
    return destination


def parse_sha256sums(path: Path) -> dict[str, str]:
    result: dict[str, str] = {}
    for raw_line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            continue
        checksum, relative_path = parts
        relative_path = relative_path.lstrip("*").lstrip("./")
        if len(checksum) == 64:
            result[relative_path] = checksum.lower()
    return result


def partition_for_subject(subject_id: int) -> str:
    if subject_id in VALIDATION_SUBJECT_IDS:
        return "validation"
    if subject_id in TRAIN_SUBJECT_IDS:
        return "train"
    raise ValueError(f"Unknown subject: {subject_id}")


def record_basename(session_id: int, subject_id: int, gesture_id: int, trial_id: int) -> str:
    return (
        f"session{session_id}_participant{subject_id}"
        f"_gesture{gesture_id}_trial{trial_id}"
    )


def record_relative_dir(session_id: int, subject_id: int) -> str:
    return f"Session{session_id}/session{session_id}_participant{subject_id}"


def record_relative_base(session_id: int, subject_id: int, gesture_id: int, trial_id: int) -> str:
    return f"{record_relative_dir(session_id, subject_id)}/{record_basename(session_id, subject_id, gesture_id, trial_id)}"


def remote_pn_dir(session_id: int, subject_id: int) -> str:
    return f"{PHYSIONET_DATABASE_DIR}/{record_relative_dir(session_id, subject_id)}"


def stable_contract_hash(payload: dict[str, Any]) -> str:
    encoded = json.dumps(json_safe(payload), sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


ETL_CONTRACT_PAYLOAD = {
    "dataset_version": DATASET_VERSION,
    "dataset_view_id": DATASET_VIEW_ID,
    "subjects": list(EXPECTED_SUBJECT_IDS),
    "sessions": list(EXPECTED_SESSION_IDS),
    "gestures": {str(k): v for k, v in GESTURE_ID_TO_CANONICAL.items()},
    "trials": list(EXPECTED_TRIAL_IDS),
    "channels": list(PRIMARY_CHANNEL_IDS),
    "channel_indices": list(PRIMARY_CHANNEL_INDICES),
    "fs_hz": EXPECTED_FS_HZ,
    "record_samples": EXPECTED_SIGNAL_SAMPLES,
    "window_samples": WINDOW_SAMPLES,
    "hop_samples": HOP_SAMPLES,
    "feature_order": list(FEATURE_ORDER),
    "preprocessing_policy_id": PREPROCESSING_POLICY_ID,
    "split_version": SPLIT_VERSION,
    "train_subjects": list(TRAIN_SUBJECT_IDS),
    "validation_subjects": list(VALIDATION_SUBJECT_IDS),
}
ETL_CONTRACT_HASH = stable_contract_hash(ETL_CONTRACT_PAYLOAD)

print(f"[PASS] Helpers initialized. ETL contract hash: {ETL_CONTRACT_HASH}")


[PASS] Helpers initialized. ETL contract hash: c17ce7945846531aac5867bfbc4148b329a1492fd03864b93ea2c4ddb303c219


## 4 — Metadata acquisition, record catalog và remote header preflight

**Input:** PhysioNet metadata endpoints.  
**Output:** catalog chính xác của 3612 record được chọn; không tải raw ZIP.


In [4]:
# === CELL 4: Fetch metadata, build record catalog and preflight remote WFDB ===
print("=" * 92)
print("[CELL 4] METADATA ACQUISITION AND RECORD CATALOG")
print("=" * 92)

small_files = {
    "RECORDS": RECORDS_PATH,
    "SHA256SUMS.txt": SHA256SUMS_PATH,
    "MotionSequence.txt": MOTION_SEQUENCE_PATH,
    "subject-info.csv": SUBJECT_INFO_PATH,
    "LICENSE.txt": LICENSE_PATH,
}

for remote_name, local_path in small_files.items():
    if not local_path.exists() or local_path.stat().st_size == 0:
        print(f"[INFO] Fetching metadata: {remote_name}")
        fetch_small_file(remote_name, local_path)
    print(f"[OK] {remote_name}: {local_path.stat().st_size:,} bytes")

records_entries = {
    line.strip().lstrip("./")
    for line in RECORDS_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    if line.strip()
}
sha256_map = parse_sha256sums(SHA256SUMS_PATH)

if not records_entries:
    raise RuntimeError("RECORDS is empty.")
if not sha256_map:
    raise RuntimeError("SHA256SUMS.txt could not be parsed.")

catalog_rows: list[dict[str, Any]] = []
missing_from_records: list[str] = []
missing_hashes: list[str] = []

for session_id in ACTIVE_SESSION_IDS:
    for subject_id in ACTIVE_SUBJECT_IDS:
        for gesture_id in ACTIVE_GESTURE_IDS:
            for trial_id in ACTIVE_TRIAL_IDS:
                base = record_relative_base(session_id, subject_id, gesture_id, trial_id)
                dat_path = f"{base}.dat"
                hea_path = f"{base}.hea"

                # PhysioNet RECORDS files may list either each concrete file
                # or only the extension-free record base. Accept both forms,
                # while SHA256SUMS must still contain each concrete .dat/.hea file.
                for expected_path in (dat_path, hea_path):
                    if expected_path not in records_entries and base not in records_entries:
                        missing_from_records.append(expected_path)
                    if expected_path not in sha256_map:
                        missing_hashes.append(expected_path)

                canonical_label = GESTURE_ID_TO_CANONICAL[gesture_id]
                partition = partition_for_subject(subject_id)
                record_id = (
                    f"grabmyo-S{subject_id:03d}-D{session_id}"
                    f"-{canonical_label}-T{trial_id:02d}"
                )
                catalog_rows.append({
                    "dataset_id": DATASET_ID,
                    "dataset_version": DATASET_VERSION,
                    "dataset_view_id": DATASET_VIEW_ID,
                    "record_id": record_id,
                    "subject_id": subject_id,
                    "subject_key": f"S{subject_id:03d}",
                    "session_id": session_id,
                    "day_id": session_id,
                    "gesture_id": gesture_id,
                    "raw_label": f"gesture{gesture_id}",
                    "canonical_label": canonical_label,
                    "trial_id": trial_id,
                    "repetition_id": record_id,
                    "partition": partition,
                    "wfdb_record_name": record_basename(session_id, subject_id, gesture_id, trial_id),
                    "wfdb_pn_dir": remote_pn_dir(session_id, subject_id),
                    "relative_base": base,
                    "dat_relative_path": dat_path,
                    "hea_relative_path": hea_path,
                    "dat_sha256": sha256_map.get(dat_path, ""),
                    "hea_sha256": sha256_map.get(hea_path, ""),
                    "expected_fs_hz": EXPECTED_FS_HZ,
                    "expected_samples": EXPECTED_SIGNAL_SAMPLES,
                    "expected_header_channels": EXPECTED_HEADER_CHANNEL_COUNT,
                    "selected_channel_indices": ";".join(map(str, PRIMARY_CHANNEL_INDICES)),
                    "selected_channel_ids": ";".join(PRIMARY_CHANNEL_IDS),
                    "expected_windows": EXPECTED_WINDOWS_PER_RECORD,
                    "etl_contract_hash": ETL_CONTRACT_HASH,
                })

if missing_from_records:
    raise RuntimeError(
        f"Selected files missing from RECORDS: {len(missing_from_records)}\n"
        + "\n".join(missing_from_records[:20])
    )
if missing_hashes:
    raise RuntimeError(
        f"Selected files missing from SHA256SUMS.txt: {len(missing_hashes)}\n"
        + "\n".join(missing_hashes[:20])
    )

catalog_df = pd.DataFrame(catalog_rows).sort_values(
    ["session_id", "subject_id", "gesture_id", "trial_id"]
).reset_index(drop=True)
write_dataframe_atomic(catalog_df, RECORD_CATALOG_CSV)

split_rows = [
    {
        "subject_id": subject_id,
        "subject_key": f"S{subject_id:03d}",
        "partition": partition_for_subject(subject_id),
        "split_version": SPLIT_VERSION,
    }
    for subject_id in EXPECTED_SUBJECT_IDS
]
split_df = pd.DataFrame(split_rows)
write_dataframe_atomic(split_df, SUBJECT_SPLIT_CSV)
write_json_atomic(
    {
        "schema_version": "grabmyo-subject-split.v1",
        "split_version": SPLIT_VERSION,
        "train_subject_ids": list(TRAIN_SUBJECT_IDS),
        "validation_subject_ids": list(VALIDATION_SUBJECT_IDS),
        "subject_overlap": [],
        "policy": "all sessions for one subject remain in one partition",
    },
    SUBJECT_SPLIT_JSON,
)

active_expected_record_count = (
    len(ACTIVE_SUBJECT_IDS) * len(ACTIVE_SESSION_IDS)
    * len(ACTIVE_GESTURE_IDS) * len(ACTIVE_TRIAL_IDS)
)
if len(catalog_df) != active_expected_record_count:
    raise RuntimeError(
        f"Catalog count mismatch: expected={active_expected_record_count}, actual={len(catalog_df)}"
    )
if catalog_df["record_id"].duplicated().any():
    raise RuntimeError("Duplicate record_id in catalog.")

# Header-only remote preflight: a few deterministic records, no signal body.
preflight_rows = catalog_df.iloc[
    sorted(set([0, len(catalog_df) // 2, len(catalog_df) - 1]))
]
header_signatures = []

for row in preflight_rows.itertuples(index=False):
    print(f"[INFO] Remote header preflight: {row.relative_base}")
    header = wfdb.rdheader(row.wfdb_record_name, pn_dir=row.wfdb_pn_dir)
    signature = {
        "fs": float(header.fs),
        "sig_len": int(header.sig_len),
        "n_sig": int(header.n_sig),
        "sig_name": list(header.sig_name or []),
        "units": list(header.units or []),
    }
    header_signatures.append(signature)

    if int(round(float(header.fs))) != EXPECTED_FS_HZ:
        raise RuntimeError(f"Unexpected fs for {row.record_id}: {header.fs}")
    if int(header.sig_len) != EXPECTED_SIGNAL_SAMPLES:
        raise RuntimeError(f"Unexpected sig_len for {row.record_id}: {header.sig_len}")
    if int(header.n_sig) != EXPECTED_HEADER_CHANNEL_COUNT:
        raise RuntimeError(f"Unexpected n_sig for {row.record_id}: {header.n_sig}")
    if max(PRIMARY_CHANNEL_INDICES) >= int(header.n_sig):
        raise RuntimeError("Selected channel index exceeds header channel count.")

# Persist small metadata and manifests to Drive.
for path in [
    RECORDS_PATH, SHA256SUMS_PATH, MOTION_SEQUENCE_PATH,
    SUBJECT_INFO_PATH, LICENSE_PATH, RECORD_CATALOG_CSV,
    SUBJECT_SPLIT_CSV, SUBJECT_SPLIT_JSON,
]:
    target_dir = DRIVE_MANIFEST_DIR
    copy_atomic(path, target_dir / path.name)

print("\n" + "=" * 92)
print("[CELL 4 SUMMARY]")
print("=" * 92)
print(f"RECORDS entries              : {len(records_entries):,}")
print(f"SHA256 entries               : {len(sha256_map):,}")
print(f"Selected record catalog      : {len(catalog_df):,}")
print(f"Train subjects               : {len(TRAIN_SUBJECT_IDS)}")
print(f"Validation subjects          : {len(VALIDATION_SUBJECT_IDS)}")
print(f"Header preflight records     : {len(header_signatures)}")
print(f"Catalog                      : {RECORD_CATALOG_CSV}")
print("[PASS] Metadata and remote-header gates passed; no raw ZIP downloaded.")


[CELL 4] METADATA ACQUISITION AND RECORD CATALOG
[INFO] Fetching metadata: RECORDS
[OK] RECORDS: 2,181,437 bytes
[INFO] Fetching metadata: SHA256SUMS.txt
[OK] SHA256SUMS.txt: 4,300,947 bytes
[INFO] Fetching metadata: MotionSequence.txt
[OK] MotionSequence.txt: 630 bytes
[INFO] Fetching metadata: subject-info.csv
[OK] subject-info.csv: 1,621 bytes
[INFO] Fetching metadata: LICENSE.txt
[OK] LICENSE.txt: 14,842 bytes
[INFO] Remote header preflight: Session1/session1_participant1/session1_participant1_gesture11_trial1
[INFO] Remote header preflight: Session2/session2_participant22/session2_participant22_gesture16_trial1
[INFO] Remote header preflight: Session3/session3_participant43/session3_participant43_gesture17_trial7

[CELL 4 SUMMARY]
RECORDS entries              : 15,351
SHA256 entries               : 30,715
Selected record catalog      : 3,612
Train subjects               : 34
Validation subjects          : 9
Header preflight records     : 3
Catalog                      : /content/d

## 5 — Feature Set 14 + parity tests

**Input:** một record đã được DC-center theo channel.  
**Output:** tensor `(windows, channels, 14)` và QC flags, cùng unit/parity test.


In [5]:
# === CELL 5: Vectorized Feature Set 14 extractor and parity tests ===
def build_window_tensor(
    centered_record: np.ndarray,
    window_samples: int,
    hop_samples: int,
) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(centered_record)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D record, got {x.shape}")
    n_samples = x.shape[0]
    starts = np.arange(
        0, n_samples - window_samples + 1, hop_samples, dtype=np.int64
    )
    offsets = np.arange(window_samples, dtype=np.int64)
    windows = x[starts[:, None] + offsets[None, :], :]
    return windows, starts


def extract_feature_set_14_batch(
    windows: np.ndarray,
    fs_hz: float,
) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(windows, dtype=np.float64)
    if x.ndim != 3:
        raise ValueError(f"Expected 3D windows, got {x.shape}")
    n_windows, n_samples, n_channels = x.shape
    if n_samples < 2:
        raise ValueError("N must be >= 2")
    if not np.isfinite(x).all():
        raise ValueError("Input contains NaN/Inf; fail closed.")

    mean_values = np.mean(x, axis=1)
    rms = np.sqrt(np.mean(np.square(x), axis=1))
    mav = np.mean(np.abs(x), axis=1)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        skewness = scipy.stats.skew(x, axis=1, bias=False, nan_policy="propagate")
        kurtosis = scipy.stats.kurtosis(
            x, axis=1, fisher=True, bias=False, nan_policy="propagate"
        )

    max_signed = np.max(x, axis=1)
    min_signed = np.min(x, axis=1)
    std_sample = np.std(x, axis=1, ddof=1)

    fft_values = np.fft.rfft(x, n=n_samples, axis=1)
    power = (np.abs(fft_values) ** 2) / float(n_samples)
    frequencies = np.fft.rfftfreq(n_samples, d=1.0 / float(fs_hz))

    spectral_min = np.min(power, axis=1)
    spectral_max = np.max(power, axis=1)
    spectral_std = np.std(power, axis=1, ddof=1)

    total_power = np.sum(power, axis=1)
    zero_power = total_power <= 0.0
    cumulative_power = np.cumsum(power, axis=1)
    half_power = total_power / 2.0
    mdf_indices = np.argmax(cumulative_power >= half_power[:, None, :], axis=1)
    mdf = frequencies[mdf_indices].astype(np.float64)

    weighted_frequency_sum = np.sum(
        power * frequencies[None, :, None], axis=1
    )
    mnf = np.full((n_windows, n_channels), np.nan, dtype=np.float64)
    np.divide(weighted_frequency_sum, total_power, out=mnf, where=~zero_power)

    probabilities = np.zeros_like(power, dtype=np.float64)
    np.divide(
        power,
        total_power[:, None, :],
        out=probabilities,
        where=~zero_power[:, None, :],
    )
    entropy_terms = np.zeros_like(probabilities)
    positive = probabilities > 0.0
    entropy_terms[positive] = -probabilities[positive] * np.log2(probabilities[positive])
    entropy = np.sum(entropy_terms, axis=1)

    mdf[zero_power] = np.nan
    mnf[zero_power] = np.nan
    entropy[zero_power] = np.nan

    features = np.stack(
        [
            rms, mav, skewness, kurtosis, max_signed, min_signed,
            std_sample, mean_values, spectral_min, spectral_max,
            spectral_std, mdf, mnf, entropy,
        ],
        axis=2,
    )

    constant_window = np.ptp(x, axis=1) == 0.0
    feature_nonfinite = ~np.isfinite(features).all(axis=2)
    qc_flags = np.full((n_windows, n_channels), "", dtype=object)

    for window_index in range(n_windows):
        for channel_index in range(n_channels):
            flags = []
            if constant_window[window_index, channel_index]:
                flags.append("constant_window")
            if zero_power[window_index, channel_index]:
                flags.append("zero_power_window")
            if feature_nonfinite[window_index, channel_index]:
                flags.append("feature_nonfinite")
            qc_flags[window_index, channel_index] = ";".join(flags)

    return features, qc_flags


def scalar_reference_feature_set_14(x: np.ndarray, fs_hz: float) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    fft_values = np.fft.rfft(x)
    power = np.abs(fft_values) ** 2 / float(n)
    frequencies = np.fft.rfftfreq(n, d=1.0 / fs_hz)
    total_power = float(np.sum(power))

    if total_power <= 0:
        mdf = mnf = entropy = np.nan
    else:
        mdf = float(frequencies[np.argmax(np.cumsum(power) >= total_power / 2.0)])
        mnf = float(np.sum(frequencies * power) / total_power)
        probabilities = power / total_power
        positive = probabilities > 0
        entropy = float(-np.sum(probabilities[positive] * np.log2(probabilities[positive])))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        skewness = float(scipy.stats.skew(x, bias=False))
        kurtosis = float(scipy.stats.kurtosis(x, fisher=True, bias=False))

    return np.array([
        np.sqrt(np.mean(x ** 2)),
        np.mean(np.abs(x)),
        skewness,
        kurtosis,
        np.max(x),
        np.min(x),
        np.std(x, ddof=1),
        np.mean(x),
        np.min(power),
        np.max(power),
        np.std(power, ddof=1),
        mdf,
        mnf,
        entropy,
    ], dtype=np.float64)


# Deterministic parity tests
rng = np.random.default_rng(5014)
test_record = rng.normal(size=(EXPECTED_SIGNAL_SAMPLES, len(PRIMARY_CHANNEL_IDS)))
test_record -= test_record.mean(axis=0, keepdims=True)
test_windows, test_starts = build_window_tensor(test_record, WINDOW_SAMPLES, HOP_SAMPLES)
test_features, test_flags = extract_feature_set_14_batch(test_windows[:3], EXPECTED_FS_HZ)

assert test_windows.shape == (
    EXPECTED_WINDOWS_PER_RECORD, WINDOW_SAMPLES, len(PRIMARY_CHANNEL_IDS)
)
assert test_features.shape == (3, len(PRIMARY_CHANNEL_IDS), len(FEATURE_ORDER))
assert test_flags.shape == (3, len(PRIMARY_CHANNEL_IDS))

for window_index in range(3):
    for channel_index in (0, 7, 15):
        reference = scalar_reference_feature_set_14(
            test_windows[window_index, :, channel_index], EXPECTED_FS_HZ
        )
        np.testing.assert_allclose(
            test_features[window_index, channel_index],
            reference,
            rtol=1e-10,
            atol=1e-12,
            equal_nan=True,
        )

constant = np.zeros((1, WINDOW_SAMPLES, 1), dtype=np.float64)
constant_features, constant_flags = extract_feature_set_14_batch(constant, EXPECTED_FS_HZ)
assert not np.isfinite(constant_features).all()
assert "constant_window" in constant_flags[0, 0]
assert "zero_power_window" in constant_flags[0, 0]

print("[PASS] Feature Set 14 parity and fail-closed QC tests passed.")
print(f"Feature order: {FEATURE_ORDER}")


[PASS] Feature Set 14 parity and fail-closed QC tests passed.
Feature order: ('rms', 'mav', 'skewness_unbiased', 'kurtosis_fisher_unbiased', 'max_signed', 'min_signed', 'std_sample_ddof1', 'mean', 'spectral_min_power', 'spectral_max_power', 'spectral_std_power_ddof1', 'mdf_hz', 'mnf_hz', 'spectral_entropy_bits')


##  6 — Remote streaming extraction theo session × subject

**Input:** record catalog và PhysioNet Internet access.  
**Output:** 129 feature shards ở full mode; mỗi shard có thể resume độc lập.



In [6]:
# === CELL 6: Resume-safe remote streaming feature extraction ===
def read_remote_record_with_retry(row: pd.Series) -> tuple[np.ndarray, Any, float]:
    last_error: Exception | None = None
    started = time.perf_counter()

    for attempt in range(1, MAX_REMOTE_ATTEMPTS + 1):
        try:
            record = wfdb.rdrecord(
                str(row["wfdb_record_name"]),
                pn_dir=str(row["wfdb_pn_dir"]),
                channels=list(PRIMARY_CHANNEL_INDICES),
                physical=True,
                return_res=32,
            )
            signal = np.asarray(record.p_signal, dtype=np.float32)
            elapsed = time.perf_counter() - started
            return signal, record, elapsed
        except Exception as exc:
            last_error = exc
            if attempt >= MAX_REMOTE_ATTEMPTS:
                break
            delay = REMOTE_BACKOFF_SECONDS[min(attempt - 1, len(REMOTE_BACKOFF_SECONDS) - 1)]
            print(
                f"[WARN] Remote read failed attempt {attempt}/{MAX_REMOTE_ATTEMPTS}: "
                f"{row['relative_base']} -> {type(exc).__name__}: {exc}. Retry in {delay}s"
            )
            time.sleep(delay)

    raise RuntimeError(
        f"Remote WFDB read failed after {MAX_REMOTE_ATTEMPTS} attempts: "
        f"{row['relative_base']}"
    ) from last_error


def save_shard_npz(path: Path, payload: dict[str, np.ndarray]) -> None:
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.unlink(missing_ok=True)
    with temporary.open("wb") as file_obj:
        np.savez_compressed(file_obj, **payload)
    temporary.replace(path)


def load_valid_shard(path: Path, expected_contract_hash: str) -> bool:
    if not path.exists():
        return False
    try:
        with np.load(path, allow_pickle=False) as data:
            if "etl_contract_hash" not in data.files:
                return False
            actual_hash = str(np.asarray(data["etl_contract_hash"]).reshape(-1)[0])
            required = {
                "X", "y", "subject_id", "session_id", "gesture_id",
                "trial_id", "record_id", "repetition_id", "window_id",
                "window_ordinal", "window_start", "window_end",
                "split_names", "etl_contract_hash",
            }
            return actual_hash == expected_contract_hash and required.issubset(data.files)
    except Exception:
        return False


catalog_df = pd.read_csv(RECORD_CATALOG_CSV)
shard_keys = sorted(
    set(zip(catalog_df["session_id"].astype(int), catalog_df["subject_id"].astype(int)))
)

print("=" * 92)
print("[CELL 6] REMOTE STREAMING EXTRACTION")
print("=" * 92)
print(f"Shards to process: {len(shard_keys)}")

for shard_number, (session_id, subject_id) in enumerate(shard_keys, start=1):
    shard_id = f"session{session_id}-subject{subject_id:03d}"
    shard_npz = SHARD_DIR / f"{shard_id}.npz"
    shard_exclusion_csv = EXCLUSION_SHARD_DIR / f"{shard_id}-exclusions.csv"
    shard_audit_csv = AUDIT_SHARD_DIR / f"{shard_id}-record-audit.csv"
    shard_evidence_json = SHARD_DIR / f"{shard_id}-evidence.json"

    persistent_shard_npz = DRIVE_SHARD_DIR / shard_npz.name
    persistent_exclusion_csv = DRIVE_EXCLUSION_SHARD_DIR / shard_exclusion_csv.name
    persistent_audit_csv = DRIVE_AUDIT_SHARD_DIR / shard_audit_csv.name
    persistent_evidence_json = DRIVE_SHARD_DIR / shard_evidence_json.name

    # Restore a valid derived checkpoint from Drive after a Colab reconnect.
    if (
        not FORCE_RECOMPUTE_SHARDS
        and not load_valid_shard(shard_npz, ETL_CONTRACT_HASH)
        and load_valid_shard(persistent_shard_npz, ETL_CONTRACT_HASH)
        and persistent_audit_csv.exists()
        and persistent_evidence_json.exists()
    ):
        copy_atomic(persistent_shard_npz, shard_npz)
        copy_atomic(persistent_audit_csv, shard_audit_csv)
        copy_atomic(persistent_evidence_json, shard_evidence_json)
        if persistent_exclusion_csv.exists():
            copy_atomic(persistent_exclusion_csv, shard_exclusion_csv)
        print(f"[{shard_number:03d}/{len(shard_keys):03d}] [RESTORE] {shard_id}")

    if (
        not FORCE_RECOMPUTE_SHARDS
        and load_valid_shard(shard_npz, ETL_CONTRACT_HASH)
        and shard_audit_csv.exists()
        and shard_evidence_json.exists()
    ):
        print(f"[{shard_number:03d}/{len(shard_keys):03d}] [SKIP] {shard_id}")
        continue

    shard_rows = catalog_df[
        (catalog_df["session_id"].astype(int) == session_id)
        & (catalog_df["subject_id"].astype(int) == subject_id)
    ].sort_values(["gesture_id", "trial_id"])

    expected_records_in_shard = len(ACTIVE_GESTURE_IDS) * len(ACTIVE_TRIAL_IDS)
    if len(shard_rows) != expected_records_in_shard:
        raise RuntimeError(
            f"Shard catalog mismatch {shard_id}: expected={expected_records_in_shard}, actual={len(shard_rows)}"
        )

    print(
        f"[{shard_number:03d}/{len(shard_keys):03d}] [RUN] {shard_id} "
        f"({len(shard_rows)} records)"
    )

    X_parts: list[np.ndarray] = []
    y_parts: list[np.ndarray] = []
    subject_parts: list[np.ndarray] = []
    session_parts: list[np.ndarray] = []
    gesture_parts: list[np.ndarray] = []
    trial_parts: list[np.ndarray] = []
    record_parts: list[np.ndarray] = []
    repetition_parts: list[np.ndarray] = []
    window_id_parts: list[np.ndarray] = []
    ordinal_parts: list[np.ndarray] = []
    start_parts: list[np.ndarray] = []
    end_parts: list[np.ndarray] = []
    split_parts: list[np.ndarray] = []

    exclusion_rows: list[dict[str, Any]] = []
    audit_rows: list[dict[str, Any]] = []
    shard_started = time.perf_counter()

    for row_tuple in shard_rows.itertuples(index=False):
        row = pd.Series(row_tuple._asdict())
        signal, wfdb_record, read_seconds = read_remote_record_with_retry(row)

        fs_hz = float(wfdb_record.fs)
        if int(round(fs_hz)) != EXPECTED_FS_HZ:
            raise RuntimeError(f"Unexpected fs: {row['record_id']} -> {fs_hz}")
        if signal.shape != (EXPECTED_SIGNAL_SAMPLES, len(PRIMARY_CHANNEL_IDS)):
            raise RuntimeError(
                f"Unexpected signal shape {row['record_id']}: {signal.shape}"
            )
        if not np.isfinite(signal).all():
            raise RuntimeError(f"Raw selected signal contains NaN/Inf: {row['record_id']}")

        centered = signal.astype(np.float64, copy=False)
        centered = centered - centered.mean(axis=0, keepdims=True)
        windows, starts = build_window_tensor(centered, WINDOW_SAMPLES, HOP_SAMPLES)
        if windows.shape[0] != EXPECTED_WINDOWS_PER_RECORD:
            raise RuntimeError(
                f"Window count mismatch {row['record_id']}: {windows.shape[0]}"
            )

        features_3d, qc_flags = extract_feature_set_14_batch(windows, fs_hz)
        valid_mask = np.isfinite(features_3d).all(axis=(1, 2))
        valid_indices = np.flatnonzero(valid_mask)
        invalid_indices = np.flatnonzero(~valid_mask)

        flattened = features_3d.reshape(
            features_3d.shape[0], len(PRIMARY_CHANNEL_IDS) * len(FEATURE_ORDER)
        )
        valid_X = flattened[valid_mask].astype(np.float32, copy=False)

        n_valid = int(valid_mask.sum())
        n_excluded = int((~valid_mask).sum())

        if n_valid > 0:
            valid_starts = starts[valid_mask].astype(np.int32)
            valid_ends = (valid_starts + WINDOW_SAMPLES).astype(np.int32)
            window_ordinals = valid_indices.astype(np.int16)
            window_ids = np.asarray(
                [f"{row['record_id']}-W{int(i):03d}" for i in window_ordinals],
                dtype="U96",
            )

            X_parts.append(valid_X)
            y_parts.append(np.full(n_valid, str(row["canonical_label"]), dtype="U32"))
            subject_parts.append(np.full(n_valid, int(row["subject_id"]), dtype=np.int16))
            session_parts.append(np.full(n_valid, int(row["session_id"]), dtype=np.int8))
            gesture_parts.append(np.full(n_valid, int(row["gesture_id"]), dtype=np.int8))
            trial_parts.append(np.full(n_valid, int(row["trial_id"]), dtype=np.int8))
            record_parts.append(np.full(n_valid, str(row["record_id"]), dtype="U96"))
            repetition_parts.append(np.full(n_valid, str(row["repetition_id"]), dtype="U96"))
            window_id_parts.append(window_ids)
            ordinal_parts.append(window_ordinals)
            start_parts.append(valid_starts)
            end_parts.append(valid_ends)
            split_parts.append(np.full(n_valid, str(row["partition"]), dtype="U16"))

        for invalid_index in invalid_indices:
            channel_flags = []
            invalid_channels = []
            for channel_index, channel_id in enumerate(PRIMARY_CHANNEL_IDS):
                flag = str(qc_flags[invalid_index, channel_index])
                if flag:
                    invalid_channels.append(channel_id)
                    channel_flags.append(f"{channel_id}:{flag}")
            exclusion_rows.append({
                "record_id": row["record_id"],
                "repetition_id": row["repetition_id"],
                "window_id": f"{row['record_id']}-W{int(invalid_index):03d}",
                "subject_id": int(row["subject_id"]),
                "session_id": int(row["session_id"]),
                "gesture_id": int(row["gesture_id"]),
                "canonical_label": row["canonical_label"],
                "trial_id": int(row["trial_id"]),
                "partition": row["partition"],
                "window_ordinal": int(invalid_index),
                "window_start": int(starts[invalid_index]),
                "window_end": int(starts[invalid_index] + WINDOW_SAMPLES),
                "reason_code": "NONFINITE_PRIMARY_FEATURE",
                "invalid_primary_channels": ";".join(invalid_channels),
                "qc_flags": ";".join(channel_flags),
                "etl_contract_hash": ETL_CONTRACT_HASH,
            })

        audit_rows.append({
            "record_id": row["record_id"],
            "repetition_id": row["repetition_id"],
            "subject_id": int(row["subject_id"]),
            "session_id": int(row["session_id"]),
            "gesture_id": int(row["gesture_id"]),
            "canonical_label": row["canonical_label"],
            "trial_id": int(row["trial_id"]),
            "partition": row["partition"],
            "relative_base": row["relative_base"],
            "dat_sha256": row["dat_sha256"],
            "hea_sha256": row["hea_sha256"],
            "read_seconds": float(read_seconds),
            "signal_samples": int(signal.shape[0]),
            "selected_channels": int(signal.shape[1]),
            "expected_windows": EXPECTED_WINDOWS_PER_RECORD,
            "valid_windows": n_valid,
            "excluded_windows": n_excluded,
            "valid_window_fraction": n_valid / EXPECTED_WINDOWS_PER_RECORD,
            "fully_excluded": n_valid == 0,
            "partially_excluded": 0 < n_valid < EXPECTED_WINDOWS_PER_RECORD,
            "status": "PASS" if n_excluded == 0 else "PASS_WITH_EXCLUSIONS",
            "etl_contract_hash": ETL_CONTRACT_HASH,
        })

        # Explicitly release the only raw signal held in memory.
        del signal, centered, windows, features_3d, qc_flags, flattened, valid_X

    if not X_parts:
        raise RuntimeError(f"No valid windows in shard: {shard_id}")

    shard_payload = {
        "X": np.concatenate(X_parts, axis=0),
        "y": np.concatenate(y_parts),
        "subject_id": np.concatenate(subject_parts),
        "session_id": np.concatenate(session_parts),
        "gesture_id": np.concatenate(gesture_parts),
        "trial_id": np.concatenate(trial_parts),
        "record_id": np.concatenate(record_parts),
        "repetition_id": np.concatenate(repetition_parts),
        "window_id": np.concatenate(window_id_parts),
        "window_ordinal": np.concatenate(ordinal_parts),
        "window_start": np.concatenate(start_parts),
        "window_end": np.concatenate(end_parts),
        "split_names": np.concatenate(split_parts),
        "etl_contract_hash": np.asarray(ETL_CONTRACT_HASH),
    }

    save_shard_npz(shard_npz, shard_payload)
    exclusion_columns = [
        "record_id", "repetition_id", "window_id", "subject_id", "session_id",
        "gesture_id", "canonical_label", "trial_id", "partition",
        "window_ordinal", "window_start", "window_end", "reason_code",
        "invalid_primary_channels", "qc_flags", "etl_contract_hash",
    ]
    exclusion_frame = pd.DataFrame(exclusion_rows, columns=exclusion_columns)
    write_dataframe_atomic(exclusion_frame, shard_exclusion_csv)
    write_dataframe_atomic(pd.DataFrame(audit_rows), shard_audit_csv)

    shard_evidence = {
        "schema_version": "grabmyo-feature-shard.v1",
        "created_at_utc": utc_now_iso(),
        "shard_id": shard_id,
        "session_id": session_id,
        "subject_id": subject_id,
        "record_count": len(audit_rows),
        "valid_window_count": int(shard_payload["X"].shape[0]),
        "excluded_window_count": len(exclusion_rows),
        "feature_dimension": int(shard_payload["X"].shape[1]),
        "elapsed_seconds": time.perf_counter() - shard_started,
        "etl_contract_hash": ETL_CONTRACT_HASH,
        "raw_files_persisted": False,
    }
    write_json_atomic(shard_evidence, shard_evidence_json)

    # Persist only derived feature/audit checkpoints; never raw WFDB files.
    copy_atomic(shard_npz, persistent_shard_npz)
    copy_atomic(shard_exclusion_csv, persistent_exclusion_csv)
    copy_atomic(shard_audit_csv, persistent_audit_csv)
    copy_atomic(shard_evidence_json, persistent_evidence_json)

    print(
        f"    [OK] valid={shard_payload['X'].shape[0]:,}, "
        f"excluded={len(exclusion_rows):,}, "
        f"elapsed={shard_evidence['elapsed_seconds']:.1f}s"
    )

print("[PASS] All active session-subject shards are available.")


[CELL 6] REMOTE STREAMING EXTRACTION
Shards to process: 129
[001/129] [RUN] session1-subject001 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=57.0s
[002/129] [RUN] session1-subject002 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=54.9s
[003/129] [RUN] session1-subject003 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=55.0s
[004/129] [RUN] session1-subject004 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=55.0s
[005/129] [RUN] session1-subject005 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=56.0s
[006/129] [RUN] session1-subject006 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=54.8s
[007/129] [RUN] session1-subject007 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=55.0s
[008/129] [RUN] session1-subject008 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=52.1s
[009/129] [RUN] session1-subject009 (28 records)
    [OK] valid=1,344, excluded=0, elapsed=52.6s
[010/129] [RUN] session1-subject010 (28 records)
    [OK] valid=1,3

## 7 — Aggregate shards, export canonical NPZ và provenance

**Input:** feature shards đã PASS.  
**Output:** NPZ đầu vào cho notebook baseline cùng coverage/exclusion/evidence manifests.


In [9]:
# === CELL 7: Aggregate shards and export canonical modeling artifact ===
print("=" * 92)
print("[CELL 7] AGGREGATE FEATURE SHARDS")
print("=" * 92)

catalog_df = pd.read_csv(RECORD_CATALOG_CSV)
shard_keys = sorted(
    set(zip(catalog_df["session_id"].astype(int), catalog_df["subject_id"].astype(int)))
)

expected_shard_paths = [
    SHARD_DIR / f"session{session_id}-subject{subject_id:03d}.npz"
    for session_id, subject_id in shard_keys
]
missing_shards = [path for path in expected_shard_paths if not load_valid_shard(path, ETL_CONTRACT_HASH)]
if missing_shards:
    raise RuntimeError(
        f"Missing/invalid feature shards: {len(missing_shards)}\n"
        + "\n".join(str(path) for path in missing_shards[:20])
    )

arrays_by_key: dict[str, list[np.ndarray]] = defaultdict(list)
required_array_keys = (
    "X", "y", "subject_id", "session_id", "gesture_id", "trial_id",
    "record_id", "repetition_id", "window_id", "window_ordinal",
    "window_start", "window_end", "split_names",
)

for path in expected_shard_paths:
    with np.load(path, allow_pickle=False) as shard:
        for key in required_array_keys:
            arrays_by_key[key].append(np.asarray(shard[key]))

combined = {key: np.concatenate(parts, axis=0) for key, parts in arrays_by_key.items()}
X = np.asarray(combined["X"], dtype=np.float32)
y = np.asarray(combined["y"]).astype("U32")
subject_id = np.asarray(combined["subject_id"], dtype=np.int16)
session_id = np.asarray(combined["session_id"], dtype=np.int8)
gesture_id = np.asarray(combined["gesture_id"], dtype=np.int8)
trial_id = np.asarray(combined["trial_id"], dtype=np.int8)
record_id = np.asarray(combined["record_id"]).astype("U96")
repetition_id = np.asarray(combined["repetition_id"]).astype("U96")
window_id = np.asarray(combined["window_id"]).astype("U128")
window_ordinal = np.asarray(combined["window_ordinal"], dtype=np.int16)
window_start = np.asarray(combined["window_start"], dtype=np.int32)
window_end = np.asarray(combined["window_end"], dtype=np.int32)
split_names = np.asarray(combined["split_names"]).astype("U16")

n_rows = X.shape[0]
for name, array in [
    ("y", y), ("subject_id", subject_id), ("session_id", session_id),
    ("gesture_id", gesture_id), ("trial_id", trial_id),
    ("record_id", record_id), ("repetition_id", repetition_id),
    ("window_id", window_id), ("window_ordinal", window_ordinal),
    ("window_start", window_start), ("window_end", window_end),
    ("split_names", split_names),
]:
    if len(array) != n_rows:
        raise RuntimeError(f"Row count mismatch for {name}: {len(array)} != {n_rows}")

if X.ndim != 2 or X.shape[1] != EXPECTED_FEATURE_DIM:
    raise RuntimeError(f"Unexpected X shape: {X.shape}")
if not np.isfinite(X).all():
    raise RuntimeError("Final X contains NaN/Inf.")
if len(window_id) != len(np.unique(window_id)):
    raise RuntimeError("Duplicate window_id in final matrix.")

# Aggregate record coverage and exclusions.
audit_paths = sorted(AUDIT_SHARD_DIR.glob("*-record-audit.csv"))
if len(audit_paths) != len(expected_shard_paths):
    raise RuntimeError(
        f"Record-audit shard count mismatch: {len(audit_paths)} != {len(expected_shard_paths)}"
    )
coverage_df = pd.concat([pd.read_csv(path) for path in audit_paths], ignore_index=True)
coverage_df = coverage_df.sort_values(
    ["session_id", "subject_id", "gesture_id", "trial_id"]
).reset_index(drop=True)
write_dataframe_atomic(coverage_df, RECORD_COVERAGE_CSV)

exclusion_paths = sorted(EXCLUSION_SHARD_DIR.glob("*-exclusions.csv"))
exclusion_frames = []
for path in exclusion_paths:
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        continue
    if not frame.empty:
        exclusion_frames.append(frame)
exclusion_df = (
    pd.concat(exclusion_frames, ignore_index=True)
    if exclusion_frames
    else pd.DataFrame(columns=[
        "record_id", "repetition_id", "window_id", "subject_id", "session_id",
        "gesture_id", "canonical_label", "trial_id", "partition",
        "window_ordinal", "window_start", "window_end", "reason_code",
        "invalid_primary_channels", "qc_flags", "etl_contract_hash",
    ])
)
write_dataframe_atomic(exclusion_df, EXCLUSION_REPORT_CSV)

active_expected_records = len(catalog_df)
active_expected_windows = active_expected_records * EXPECTED_WINDOWS_PER_RECORD
if len(coverage_df) != active_expected_records:
    raise RuntimeError(
        f"Coverage record count mismatch: expected={active_expected_records}, actual={len(coverage_df)}"
    )
if int(coverage_df["expected_windows"].sum()) != active_expected_windows:
    raise RuntimeError("Coverage expected-window sum mismatch.")
if int(coverage_df["valid_windows"].sum()) != n_rows:
    raise RuntimeError("Coverage valid-window sum does not match X rows.")
if int(coverage_df["excluded_windows"].sum()) != len(exclusion_df):
    raise RuntimeError("Coverage exclusion sum does not match exclusion report.")
if int(coverage_df["valid_windows"].sum() + coverage_df["excluded_windows"].sum()) != active_expected_windows:
    raise RuntimeError("Window population cannot be reconciled.")

actual_subjects = set(subject_id.astype(int).tolist())
expected_active_subjects = set(ACTIVE_SUBJECT_IDS)
if actual_subjects != expected_active_subjects:
    raise RuntimeError(
        f"Subject population mismatch: expected={sorted(expected_active_subjects)}, actual={sorted(actual_subjects)}"
    )

train_subjects_observed = set(subject_id[split_names == "train"].astype(int).tolist())
validation_subjects_observed = set(subject_id[split_names == "validation"].astype(int).tolist())
if not train_subjects_observed.isdisjoint(validation_subjects_observed):
    raise RuntimeError("Subject leakage across train/validation.")

feature_names = np.asarray(
    [f"{channel_id}__{feature_name}" for channel_id in PRIMARY_CHANNEL_IDS for feature_name in FEATURE_ORDER],
    dtype="U96",
)
channel_names = np.asarray(PRIMARY_CHANNEL_IDS, dtype="U16")
class_order = np.asarray(PRIMARY_CLASS_ORDER, dtype="U32")


def feature_arm_indices(feature_subset: Iterable[str]) -> np.ndarray:
    subset = set(feature_subset)
    indices = [
        channel_index * len(FEATURE_ORDER) + feature_index
        for channel_index in range(len(PRIMARY_CHANNEL_IDS))
        for feature_index, feature_name in enumerate(FEATURE_ORDER)
        if feature_name in subset
    ]
    return np.asarray(indices, dtype=np.int32)


f_td8_indices = feature_arm_indices(TD8_FEATURES)
f_sp6_indices = feature_arm_indices(SP6_FEATURES)
f_all14_indices = feature_arm_indices(FEATURE_ORDER)
f_no_moments12_indices = feature_arm_indices(NO_MOMENTS12_FEATURES)

column_manifest = {
    "schema_version": "grabmyo-feature-column-manifest.v1",
    "created_at_utc": utc_now_iso(),
    "dataset_id": DATASET_ID,
    "dataset_version": DATASET_VERSION,
    "dataset_view_id": DATASET_VIEW_ID,
    "channel_policy_id": CHANNEL_POLICY_ID,
    "channel_order": list(PRIMARY_CHANNEL_IDS),
    "feature_set_version": FEATURE_SET_VERSION,
    "feature_order_per_channel": list(FEATURE_ORDER),
    "flattening_order": "channel-major then feature-order",
    "feature_names": feature_names.tolist(),
    "feature_arms": {
        "F-TD8": f_td8_indices.tolist(),
        "F-SP6": f_sp6_indices.tolist(),
        "F-ALL14": f_all14_indices.tolist(),
        "F-NO-MOMENTS12": f_no_moments12_indices.tolist(),
    },
}
write_json_atomic(column_manifest, COLUMN_MANIFEST_JSON)

npz_temporary = FINAL_NPZ.with_suffix(FINAL_NPZ.suffix + ".part")
npz_temporary.unlink(missing_ok=True)
with npz_temporary.open("wb") as file_obj:
    np.savez_compressed(
        file_obj,
        X=X,
        y=y,
        groups=subject_id,
        subject_id=subject_id,
        subject_key=np.asarray([f"S{int(v):03d}" for v in subject_id], dtype="U16"),
        session_id=session_id,
        day_id=session_id,
        gesture_id=gesture_id,
        trial_id=trial_id,
        record_id=record_id,
        repetition_id=repetition_id,
        window_id=window_id,
        window_ordinal=window_ordinal,
        window_start=window_start,
        window_end=window_end,
        split_names=split_names,
        feature_names=feature_names,
        channel_names=channel_names,
        class_order=class_order,
        F_TD8_indices=f_td8_indices,
        F_SP6_indices=f_sp6_indices,
        F_ALL14_indices=f_all14_indices,
        F_NO_MOMENTS12_indices=f_no_moments12_indices,
        dataset_id=np.asarray(DATASET_ID),
        dataset_version=np.asarray(DATASET_VERSION),
        dataset_view_id=np.asarray(DATASET_VIEW_ID),
        feature_set_version=np.asarray(FEATURE_SET_VERSION),
        channel_policy_id=np.asarray(CHANNEL_POLICY_ID),
        preprocessing_policy_id=np.asarray(PREPROCESSING_POLICY_ID),
        windowing_version=np.asarray(WINDOWING_VERSION),
        split_version=np.asarray(SPLIT_VERSION),
        label_mapping_version=np.asarray(LABEL_MAPPING_VERSION),
        etl_contract_hash=np.asarray(ETL_CONTRACT_HASH),
        source_record_catalog_sha256=np.asarray(sha256_file(RECORD_CATALOG_CSV)),
        training_allowed=np.asarray(False),
        model_fitting_allowed=np.asarray(False),
        scaler_fitting_allowed=np.asarray(False),
        test_set_opened=np.asarray(False),
        pooled_training_allowed=np.asarray(False),
        clinical_use_allowed=np.asarray(False),
    )
npz_temporary.replace(FINAL_NPZ)

npz_sha256 = sha256_file(FINAL_NPZ)

def parse_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    normalized = series.astype(str).str.strip().str.lower()
    allowed = {"true", "false", "1", "0"}
    unexpected = set(normalized.unique()) - allowed
    if unexpected:
        raise RuntimeError(f"Cannot parse boolean values: {sorted(unexpected)}")
    return normalized.isin({"true", "1"})

coverage_df["fully_excluded"] = parse_bool_series(coverage_df["fully_excluded"])
coverage_df["partially_excluded"] = parse_bool_series(coverage_df["partially_excluded"])
fully_excluded_records = coverage_df[coverage_df["fully_excluded"]]
partially_excluded_records = coverage_df[coverage_df["partially_excluded"]]

etl_evidence = {
    "schema_version": "grabmyo-remote-stream-etl-evidence.v1",
    "created_at_utc": utc_now_iso(),
    "run_mode": RUN_MODE,
    "dataset": {
        "dataset_id": DATASET_ID,
        "source": DATASET_SOURCE,
        "version": DATASET_VERSION,
        "doi": DATASET_DOI,
        "dataset_view_id": DATASET_VIEW_ID,
    },
    "source_access": {
        "strategy": "wfdb remote streaming from PhysioNet",
        "zip_downloaded": False,
        "raw_records_persisted": False,
        "selected_record_count": active_expected_records,
        "selected_channel_count": len(PRIMARY_CHANNEL_IDS),
    },
    "contracts": ETL_CONTRACT_PAYLOAD,
    "contract_hash": ETL_CONTRACT_HASH,
    "population": {
        "expected_records": active_expected_records,
        "processed_records": len(coverage_df),
        "expected_windows_before_qc": active_expected_windows,
        "valid_windows": n_rows,
        "excluded_windows": len(exclusion_df),
        "fully_excluded_records": len(fully_excluded_records),
        "partially_excluded_records": len(partially_excluded_records),
        "subjects": sorted(actual_subjects),
        "sessions": sorted(set(session_id.astype(int).tolist())),
        "class_counts_windows": {str(k): int(v) for k, v in Counter(y.tolist()).items()},
    },
    "governance": {
        "training_allowed": TRAINING_ALLOWED,
        "model_fitting_allowed": MODEL_FITTING_ALLOWED,
        "scaler_fitting_allowed": SCALER_FITTING_ALLOWED,
        "test_set_opened": TEST_SET_OPENED,
        "clinical_use_allowed": CLINICAL_USE_ALLOWED,
    },
    "artifacts": {
        "npz": str(FINAL_NPZ),
        "npz_sha256": npz_sha256,
        "record_catalog": str(RECORD_CATALOG_CSV),
        "record_coverage": str(RECORD_COVERAGE_CSV),
        "exclusion_report": str(EXCLUSION_REPORT_CSV),
        "column_manifest": str(COLUMN_MANIFEST_JSON),
    },
}
write_json_atomic(etl_evidence, ETL_EVIDENCE_JSON)

# Keep pass/fail checks strictly boolean.
# Numeric diagnostics belong in a separate counts section.
duplicate_window_id_count = int(
    len(window_id) - len(np.unique(window_id))
)

final_gate = {
    "schema_version": "grabmyo-etl-final-gate.v1.1",
    "created_at_utc": utc_now_iso(),
    "status": (
        "PASS"
        if len(exclusion_df) == 0
        else "PASS_WITH_DOCUMENTED_EXCLUSIONS"
    ),
    "run_mode": RUN_MODE,
    "checks": {
        "all_active_records_processed": bool(
            len(coverage_df) == active_expected_records
        ),
        "window_population_reconciled": bool(
            n_rows + len(exclusion_df) == active_expected_windows
        ),
        "finite_feature_matrix": bool(np.isfinite(X).all()),
        "feature_dimension_correct": bool(
            X.shape[1] == EXPECTED_FEATURE_DIM
        ),
        "subject_split_disjoint": bool(
            train_subjects_observed.isdisjoint(
                validation_subjects_observed
            )
        ),
        "no_duplicate_window_ids": bool(
            duplicate_window_id_count == 0
        ),
        "raw_zip_not_downloaded": True,
        "raw_records_not_persisted": True,
    },
    "counts": {
        "duplicate_window_id_count": duplicate_window_id_count,
        "active_expected_records": int(active_expected_records),
        "processed_records": int(len(coverage_df)),
        "active_expected_windows": int(active_expected_windows),
        "valid_windows": int(n_rows),
        "excluded_windows": int(len(exclusion_df)),
    },
    "npz_sha256": npz_sha256,
    "fully_excluded_record_ids": (
        fully_excluded_records["record_id"].astype(str).tolist()
    ),
    "partially_excluded_record_ids": (
        partially_excluded_records["record_id"].astype(str).tolist()
    ),
}

invalid_check_types = {
    name: type(value).__name__
    for name, value in final_gate["checks"].items()
    if not isinstance(value, (bool, np.bool_))
}
if invalid_check_types:
    raise RuntimeError(
        "Final gate contains non-boolean check values: "
        + json.dumps(invalid_check_types, indent=2)
    )

# Normalize NumPy boolean scalars before JSON serialization.
final_gate["checks"] = {
    name: bool(value)
    for name, value in final_gate["checks"].items()
}

failed_checks = [
    name
    for name, passed in final_gate["checks"].items()
    if not passed
]

if failed_checks:
    raise RuntimeError(
        "GRABMyo ETL final gate failed checks: "
        f"{failed_checks}\n"
        + json.dumps(json_safe(final_gate), indent=2)
    )


write_json_atomic(final_gate, FINAL_GATE_JSON)

for artifact in [
    RECORD_CATALOG_CSV, SUBJECT_SPLIT_CSV, SUBJECT_SPLIT_JSON,
    RECORD_COVERAGE_CSV, EXCLUSION_REPORT_CSV, ETL_EVIDENCE_JSON, FINAL_GATE_JSON,
]:
    copy_atomic(artifact, DRIVE_MANIFEST_DIR / artifact.name)
for artifact in [FINAL_NPZ, COLUMN_MANIFEST_JSON]:
    copy_atomic(artifact, DRIVE_OUTPUT_DIR / artifact.name)

print("\n" + "=" * 92)
print("[CELL 7 SUMMARY]")
print("=" * 92)
print(f"X shape                       : {X.shape}")
print(f"Records processed             : {len(coverage_df):,}")
print(f"Valid windows                 : {n_rows:,}")
print(f"Excluded windows              : {len(exclusion_df):,}")
print(f"Fully excluded records        : {len(fully_excluded_records):,}")
print(f"Partially excluded records    : {len(partially_excluded_records):,}")
print(f"Train subjects observed       : {len(train_subjects_observed)}")
print(f"Validation subjects observed  : {len(validation_subjects_observed)}")
print(f"NPZ SHA-256                   : {npz_sha256}")
print(f"Runtime NPZ                   : {FINAL_NPZ}")
print(f"Drive NPZ                     : {DRIVE_OUTPUT_DIR / FINAL_NPZ.name}")
print(f"Final status                  : {final_gate['status']}")
print("[PASS] Canonical GRABMyo modeling artifact exported.")


[CELL 7] AGGREGATE FEATURE SHARDS

[CELL 7 SUMMARY]
X shape                       : (173376, 224)
Records processed             : 3,612
Valid windows                 : 173,376
Excluded windows              : 0
Fully excluded records        : 0
Partially excluded records    : 0
Train subjects observed       : 34
Validation subjects observed  : 9
NPZ SHA-256                   : cbd4e0f7203d85c7acb81ea985b436675eb451022fcd11bacb6f3d0433f5a197
Runtime NPZ                   : /content/data/grabmyo-physionet-v1.1.0/outputs/grabmyo-primary4-forearm16-fall14.npz
Drive NPZ                     : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/grabmyo-primary4-forearm16-fall14.npz
Final status                  : PASS
[PASS] Canonical GRABMyo modeling artifact exported.


## CELL 8 — Reload final artifact, verify Drive copy và optional download

**Input:** NPZ vừa xuất.  
**Output:** xác nhận artifact có thể được mở độc lập bởi notebook baseline.


In [10]:
# === CELL 8: Final reload and Drive verification ===
required_npz_keys = {
    "X", "y", "groups", "subject_id", "session_id", "gesture_id",
    "trial_id", "record_id", "repetition_id", "window_id",
    "window_ordinal", "window_start", "window_end", "split_names",
    "feature_names", "channel_names", "class_order",
    "F_TD8_indices", "F_SP6_indices", "F_ALL14_indices",
    "F_NO_MOMENTS12_indices", "dataset_view_id", "etl_contract_hash",
}

with np.load(FINAL_NPZ, allow_pickle=False) as final_data:
    missing_keys = required_npz_keys - set(final_data.files)
    if missing_keys:
        raise RuntimeError(f"Final NPZ missing keys: {sorted(missing_keys)}")
    X_reload = np.asarray(final_data["X"])
    y_reload = np.asarray(final_data["y"]).astype(str)
    groups_reload = np.asarray(final_data["groups"])
    split_reload = np.asarray(final_data["split_names"]).astype(str)
    view_reload = str(np.asarray(final_data["dataset_view_id"]).reshape(-1)[0])
    contract_reload = str(np.asarray(final_data["etl_contract_hash"]).reshape(-1)[0])

if view_reload != DATASET_VIEW_ID:
    raise RuntimeError(f"Dataset view mismatch: {view_reload}")
if contract_reload != ETL_CONTRACT_HASH:
    raise RuntimeError("ETL contract hash mismatch on reload.")
if X_reload.shape[0] != len(y_reload) or len(y_reload) != len(groups_reload):
    raise RuntimeError("Reloaded row alignment failed.")
if not np.isfinite(X_reload).all():
    raise RuntimeError("Reloaded X contains NaN/Inf.")

runtime_hash = sha256_file(FINAL_NPZ)
drive_npz = DRIVE_OUTPUT_DIR / FINAL_NPZ.name
if not drive_npz.exists():
    raise FileNotFoundError(f"Persistent NPZ missing: {drive_npz}")
drive_hash = sha256_file(drive_npz)
if runtime_hash != drive_hash:
    raise RuntimeError("Runtime and Drive NPZ SHA-256 mismatch.")

print("=" * 92)
print("[FINAL ETL GATE]")
print("=" * 92)
print(f"NPZ keys                  : {len(required_npz_keys)} required, all present")
print(f"X shape                   : {X_reload.shape}")
print(f"Classes                   : {sorted(np.unique(y_reload).tolist())}")
print(f"Subjects                  : {len(np.unique(groups_reload))}")
print(f"Partitions                : {Counter(split_reload.tolist())}")
print(f"Runtime/Drive SHA match   : {runtime_hash == drive_hash}")
print(f"Raw ZIP downloaded        : False")
print(f"Raw WFDB persisted        : False")
print("[PASS] GRABMyo ETL notebook completed.")

if IN_COLAB and DOWNLOAD_FINAL_NPZ_TO_BROWSER:
    files.download(str(drive_npz))


[FINAL ETL GATE]
NPZ keys                  : 23 required, all present
X shape                   : (173376, 224)
Classes                   : ['hand_close', 'rest', 'wrist_extension', 'wrist_flexion']
Subjects                  : 43
Partitions                : Counter({'train': 137088, 'validation': 36288})
Runtime/Drive SHA match   : True
Raw ZIP downloaded        : False
Raw WFDB persisted        : False
[PASS] GRABMyo ETL notebook completed.


# Expected full-mode output

```text
Records                       : 3612
Windows before QC             : 173376
Feature dimension             : 224
Subjects                      : 43
Train / validation subjects   : 34 / 9
Classes                       : 4
Raw ZIP downloaded            : False
Raw WFDB files persisted      : False
```

Notebook baseline sẽ đọc trực tiếp:

```text
/content/drive/MyDrive/MyoLab-AI-data/
└── grabmyo-physionet-v1.1.0/
    └── outputs/
        └── grabmyo-primary4-forearm16-fall14.npz
```
